# Data Preprocessing

This notebook prepares the obesity dataset for machine learning models.

The preprocessing workflow includes:

- Feature and target separation
- Numerical feature preprocessing
- Ordinal feature encoding
- Nominal feature encoding
- Train/validation/test split
- ColumnTransformer pipeline creation

The final output will be a machine learning ready dataset.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    OrdinalEncoder
)

import warnings
warnings.filterwarnings("ignore")

In [2]:
DATA_PATH = "../data/raw/obesity.csv"

df = pd.read_csv(DATA_PATH)

df.head()

,id,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,0,Male,24.443011,1.699998,81.669950,yes,yes,2.000000,2.983297,Sometimes,no,2.763573,no,0.000000,0.976473,Sometimes,Public_Transportation,Overweight_Level_II
1,1,Female,18.000000,1.560000,57.000000,yes,yes,2.000000,3.000000,Frequently,no,2.000000,no,1.000000,1.000000,no,Automobile,Normal_Weight
2,2,Female,18.000000,1.711460,50.165754,yes,yes,1.880534,1.411685,Sometimes,no,1.910378,no,0.866045,1.673584,no,Public_Transportation,Insufficient_Weight
3,3,Female,20.952737,1.710730,131.274851,yes,yes,3.000000,3.000000,Sometimes,no,1.674061,no,1.467863,0.780199,Sometimes,Public_Transportation,Obesity_Type_III
4,4,Male,31.641081,1.914186,93.798055,yes,yes,2.679664,1.971472,Sometimes,no,1.979848,no,1.967973,0.931721,Sometimes,Public_Transportation,Overweight_Level_II


In [3]:
print("Dataset Shape:", df.shape)

df.info()

Dataset Shape: (20758, 18)
<class 'pandas.DataFrame'>
RangeIndex: 20758 entries, 0 to 20757
Data columns (total 18 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              20758 non-null  int64  
 1   Gender                          20758 non-null  str    
 2   Age                             20758 non-null  float64
 3   Height                          20758 non-null  float64
 4   Weight                          20758 non-null  float64
 5   family_history_with_overweight  20758 non-null  str    
 6   FAVC                            20758 non-null  str    
 7   FCVC                            20758 non-null  float64
 8   NCP                             20758 non-null  float64
 9   CAEC                            20758 non-null  str    
 10  SMOKE                           20758 non-null  str    
 11  CH2O                            20758 non-null  float64
 12  SCC             

In [4]:
target_column = "NObeyesdad"
identifier_column = "id"

X = df.drop(
    columns=[
        identifier_column,
        target_column
    ]
)

y = df[target_column]

In [5]:
numerical_features = [
    "Age",
    "Height",
    "Weight",
    "FCVC",
    "NCP",
    "CH2O",
    "FAF",
    "TUE"
]

numerical_features

['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']

In [6]:
ordinal_features = [
    "CAEC",
    "CALC"
]

ordinal_features

['CAEC', 'CALC']

In [7]:
nominal_features = [
    "Gender",
    "family_history_with_overweight",
    "FAVC",
    "SMOKE",
    "SCC",
    "MTRANS"
]

nominal_features

['Gender', 'family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC', 'MTRANS']

In [8]:
all_features = (
    numerical_features +
    ordinal_features +
    nominal_features
)

print("Total Features:", len(all_features))
print("Dataset Features:", X.shape[1])

print(
    "All features covered:",
    set(all_features) == set(X.columns)
)

Total Features: 16
Dataset Features: 16
All features covered: True


In [9]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)


X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

In [10]:
print("Training set:", X_train.shape)
print("Validation set:", X_valid.shape)
print("Testing set:", X_test.shape)

Training set: (14530, 16)
Validation set: (3114, 16)
Testing set: (3114, 16)


In [11]:
numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [12]:
ordinal_categories = [
    ["no", "Sometimes", "Frequently", "Always"],
    ["no", "Sometimes", "Frequently"]
]

In [13]:
ordinal_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OrdinalEncoder(
                categories=ordinal_categories
            )
        )
    ]
)

In [14]:
nominal_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "ordinal",
            ordinal_pipeline,
            ordinal_features
        ),
        (
            "nominal",
            nominal_pipeline,
            nominal_features
        )
    ]
)

In [16]:
X_train_processed = preprocessor.fit_transform(X_train)

X_valid_processed = preprocessor.transform(X_valid)

X_test_processed = preprocessor.transform(X_test)

In [17]:
print("Training data shape:", X_train_processed.shape)
print("Validation data shape:", X_valid_processed.shape)
print("Testing data shape:", X_test_processed.shape)

Training data shape: (14530, 25)
Validation data shape: (3114, 25)
Testing data shape: (3114, 25)


In [18]:
feature_names = preprocessor.get_feature_names_out()

print("Total processed features:", len(feature_names))

feature_names[:20]

Total processed features: 25


array(['numerical__Age', 'numerical__Height', 'numerical__Weight',
       'numerical__FCVC', 'numerical__NCP', 'numerical__CH2O',
       'numerical__FAF', 'numerical__TUE', 'ordinal__CAEC',
       'ordinal__CALC', 'nominal__Gender_Female', 'nominal__Gender_Male',
       'nominal__family_history_with_overweight_no',
       'nominal__family_history_with_overweight_yes', 'nominal__FAVC_no',
       'nominal__FAVC_yes', 'nominal__SMOKE_no', 'nominal__SMOKE_yes',
       'nominal__SCC_no', 'nominal__SCC_yes'], dtype=object)

In [19]:
X_train_processed_df = pd.DataFrame(
    X_train_processed.toarray()
    if hasattr(X_train_processed, "toarray")
    else X_train_processed,
    columns=feature_names
)

X_train_processed_df.head()

,numerical__Age,numerical__Height,numerical__Weight,numerical__FCVC,numerical__NCP,numerical__CH2O,numerical__FAF,numerical__TUE,ordinal__CAEC,ordinal__CALC,...,nominal__FAVC_yes,nominal__SMOKE_no,nominal__SMOKE_yes,nominal__SCC_no,nominal__SCC_yes,nominal__MTRANS_Automobile,nominal__MTRANS_Bike,nominal__MTRANS_Motorbike,nominal__MTRANS_Public_Transportation,nominal__MTRANS_Walking
0,0.385914,-0.881557,0.910609,1.038204,0.343418,1.236961,-1.167175,-0.820903,1.0,1.0,...,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.373562,0.180375,0.948677,1.038204,0.343418,1.161405,-1.122673,-0.771579,1.0,1.0,...,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,1.408144,0.638226,1.215992,0.976776,0.343418,0.645981,-0.024391,-1.026618,1.0,1.0,...,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
3,0.213334,0.889680,1.006426,-2.514706,0.343418,-0.042607,-0.764289,-1.011249,1.0,1.0,...,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0.031084,-0.799056,-0.146914,1.038204,-2.476780,1.592657,0.029060,-1.026618,1.0,1.0,...,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [20]:
print("Processed training shape:")
print(X_train_processed_df.shape)

print("\nMissing values:")
print(X_train_processed_df.isnull().sum().sum())

Processed training shape:
(14530, 25)

Missing values:
0


In [21]:
print("Preprocessing completed successfully")

Preprocessing completed successfully
